# Top 50 Movies on IMDB (Kaggle Dataset)

### Goal: Sum the relevant information of movies, directors, writers, etc. into one dataframe for viewing/processing

In [13]:
import numpy as np
import pandas as pd
import os

In [7]:
import kagglehub

In [8]:
path = kagglehub.dataset_download("julianorozco/top-50-movies-by-imdb")

In [9]:
print("Path to dataset: ", path)

Path to dataset:  /Users/mayihavecrumbofcoochieplz/.cache/kagglehub/datasets/julianorozco/top-50-movies-by-imdb/versions/1


In [14]:
print(os.listdir(path))

['movies_producers.csv', 'top cast.csv', 'movies_directors.csv', 'writers.csv', 'genres.csv', 'producers.csv', 'movies_writers.csv', 'directors.csv', 'movies_genres.csv', 'movies.csv']


In [27]:
# Load csv's into Dataframes

movies = pd.read_csv(path + '/movies.csv')
directors = pd.read_csv(path + '/directors.csv')
writers = pd.read_csv(path + '/writers.csv')
top_cast = pd.read_csv(path + '/top cast.csv')
producers = pd.read_csv(path + '/producers.csv')
genres = pd.read_csv(path + '/genres.csv')
movies_genres = pd.read_csv(path + '/movies_genres.csv')

In [30]:
producers['producers'] = producers['producers'].str.strip()
producers

,id,producers
0,1,Niki Marvin
1,2,Albert S. Ruddy
2,3,"Emma Thomas, Charles Roven, Christopher Nolan"
3,4,"Francis Ford Coppola, Fred Roos"
4,5,"Henry Fonda, Reginald Rose"
5,6,"Steven Spielberg, Gerald R. Molen, Branko Lustig"
6,7,"Barrie M. Osborne, Peter Jackson, Fran Walsh"
7,8,Lawrence Bender
8,9,"Barrie M. Osborne, Peter Jackson, Fran Walsh, ..."
9,10,Alberto Grimaldi


In [31]:
# Matching id's to movies column names for merging

directors = directors.rename(columns={'id': 'Director_id'})
writers   = writers.rename(columns={'id': 'Writers_id'})
top_cast  = top_cast.rename(columns={'id': 'TopCast_id'})
producers = producers.rename(columns={'id': 'producers_id'})

In [48]:
# Merging dfs

df = (movies
      .merge(directors, on='Director_id', how='left')
      .merge(writers,   on='Writers_id',  how='left')
      .merge(top_cast,  on='TopCast_id',  how='left')
      .merge(producers, on='producers_id', how='left'))

In [51]:
# Merging genres

mg = (movies_genres
      .merge(genres, left_on='genre_id', right_on='id', how='left')
      .groupby('movie_id')['genre']
      .apply(lambda s: ', '.join(sorted(s.dropna().unique())))
      .reset_index()
      .rename(columns={'movie_id': 'id', 'genre': 'Genres'}))

df = df.drop(columns=['Genres'], errors='ignore').merge(mg, on='id', how='left')
df

,id,Top_rated,name_movie,Year,Director_id,Writers_id,TopCast_id,producers_id,Director,Writers,Top_actors,producers,Genres
0,1,1,The Shawshank Redemption,1994,1,1,1,1,Frank Dabaront,"Frank Darabont, Stephen King","Tim Robbins, Morgan Freeman, Bob Gunton, James...",Niki Marvin,Drama
1,2,2,The Godfather,1972,2,2,2,2,Francis Ford Coppola,"Francis Ford Coppola, Mario Puzo","Marlon Brando, Al Pacino, James Caan, Robert D...",Albert S. Ruddy,"Crime, Drama"
2,3,3,The Dark Knight,2008,3,3,3,3,Christopher Nolan,"Jonathan Nolan, Christopher Nolan","Christian Bale, Heath Ledger, Aaron Eckhart, M...","Emma Thomas, Charles Roven, Christopher Nolan","Action, Crime, Drama"
3,4,4,The Godfather: Part II,1974,2,2,4,4,Francis Ford Coppola,"Francis Ford Coppola, Mario Puzo","Al Pacino, Robert De Niro, Diane Keaton, Rober...","Francis Ford Coppola, Fred Roos","Crime, Drama"
4,5,5,12 Angry Men,1957,4,4,5,5,Sidney Lumet,Reginald Rose,"Henry Fonda, Lee J. Cobb, Jack Warden, E.G. Ma...","Henry Fonda, Reginald Rose","Crime, Drama"
5,6,6,Schindler's List,1993,5,5,6,6,Steven Spielberg,Steven Zaillian,"Liam Neeson, Ben Kingsley, Ralph Fiennes, Caro...","Steven Spielberg, Gerald R. Molen, Branko Lustig","Biography, Drama, History"
6,7,7,The Lord of the Rings: The Return of the King,2003,6,6,7,7,Peter Jackson,"Fran Walsh, Philippa Boyens, Peter Jackson","Elijah Wood, Viggo Mortensen, Ian McKellen, Se...","Barrie M. Osborne, Peter Jackson, Fran Walsh","Action, Adventure, Drama"
7,8,8,Pulp Fiction,1994,7,7,8,8,Quentin Tarantino,"Quentin Tarantino, Roger Avary","John Travolta, Samuel L. Jackson, Uma Thurman,...",Lawrence Bender,"Crime, Drama"
8,9,9,The Lord of the Rings: The Fellowship of the Ring,2001,6,6,9,9,Peter Jackson,"Fran Walsh, Philippa Boyens, Peter Jackson","Elijah Wood, Ian McKellen, Viggo Mortensen, Se...","Barrie M. Osborne, Peter Jackson, Fran Walsh, ...","Action, Adventure, Drama"
9,10,10,"The Good, the Bad and the Ugly",1966,8,8,10,10,Sergio Leone,"Age & Scarpelli, Luciano Vincenzoni, Sergio Leone","Clint Eastwood, Lee Van Cleef, Eli Wallach, Al...",Alberto Grimaldi,"Adventure, Western"


In [54]:
# Dropping director_id, writers_id, etc.

df_flat = (df
           .drop(columns=['Director_id', 'Writers_id', 'TopCast_id', 'producers_id'])
           [['id', 'Top_rated', 'name_movie', 'Year', 'Genres',
             'Director', 'Writers', 'producers', 'Top_actors']])

df_flat.to_csv('movies_combined.csv', index=False)
df_flat

,id,Top_rated,name_movie,Year,Genres,Director,Writers,producers,Top_actors
0,1,1,The Shawshank Redemption,1994,Drama,Frank Dabaront,"Frank Darabont, Stephen King",Niki Marvin,"Tim Robbins, Morgan Freeman, Bob Gunton, James..."
1,2,2,The Godfather,1972,"Crime, Drama",Francis Ford Coppola,"Francis Ford Coppola, Mario Puzo",Albert S. Ruddy,"Marlon Brando, Al Pacino, James Caan, Robert D..."
2,3,3,The Dark Knight,2008,"Action, Crime, Drama",Christopher Nolan,"Jonathan Nolan, Christopher Nolan","Emma Thomas, Charles Roven, Christopher Nolan","Christian Bale, Heath Ledger, Aaron Eckhart, M..."
3,4,4,The Godfather: Part II,1974,"Crime, Drama",Francis Ford Coppola,"Francis Ford Coppola, Mario Puzo","Francis Ford Coppola, Fred Roos","Al Pacino, Robert De Niro, Diane Keaton, Rober..."
4,5,5,12 Angry Men,1957,"Crime, Drama",Sidney Lumet,Reginald Rose,"Henry Fonda, Reginald Rose","Henry Fonda, Lee J. Cobb, Jack Warden, E.G. Ma..."
5,6,6,Schindler's List,1993,"Biography, Drama, History",Steven Spielberg,Steven Zaillian,"Steven Spielberg, Gerald R. Molen, Branko Lustig","Liam Neeson, Ben Kingsley, Ralph Fiennes, Caro..."
6,7,7,The Lord of the Rings: The Return of the King,2003,"Action, Adventure, Drama",Peter Jackson,"Fran Walsh, Philippa Boyens, Peter Jackson","Barrie M. Osborne, Peter Jackson, Fran Walsh","Elijah Wood, Viggo Mortensen, Ian McKellen, Se..."
7,8,8,Pulp Fiction,1994,"Crime, Drama",Quentin Tarantino,"Quentin Tarantino, Roger Avary",Lawrence Bender,"John Travolta, Samuel L. Jackson, Uma Thurman,..."
8,9,9,The Lord of the Rings: The Fellowship of the Ring,2001,"Action, Adventure, Drama",Peter Jackson,"Fran Walsh, Philippa Boyens, Peter Jackson","Barrie M. Osborne, Peter Jackson, Fran Walsh, ...","Elijah Wood, Ian McKellen, Viggo Mortensen, Se..."
9,10,10,"The Good, the Bad and the Ugly",1966,"Adventure, Western",Sergio Leone,"Age & Scarpelli, Luciano Vincenzoni, Sergio Leone",Alberto Grimaldi,"Clint Eastwood, Lee Van Cleef, Eli Wallach, Al..."
